# Independent check with official USGS GNIS canyon names

Compares the 325-canyon catalogue with the named tributaries of the Grand Canyon in the U.S. Geological Survey Geographic Names Information System (GNIS). The names were never used in training. Outputs go to `results/gnis_validation/`.

*A named valley counts as a Grand Canyon tributary when its mouth lies at least 0.5 D (614 m) below the plateau in `results/mars_analog/grand_canyon_incision_depth.tif`. It counts as found when a detection lies within 500 m of its mouth-to-head line.*

In [ ]:
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
(REPO_DIR / 'results/gnis_validation').mkdir(parents=True, exist_ok=True)
print('Repository:', REPO_DIR)


## 1. Download GNIS, select named tributaries, compare and name the detections

In [ ]:
# Step V1: independent check with official USGS GNIS canyon names (no model change)
import io
import json
import re
import zipfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.transform import rowcol
from scipy.ndimage import maximum_filter
from shapely.geometry import LineString, Point

PROJECT = REPO_DIR
DETECTIONS = PROJECT / 'results/grand_canyon/grand_canyon_detections_geology.geojson'
DEPTH = PROJECT / 'results/mars_analog/grand_canyon_incision_depth.tif'   # depth below the plateau, 200 m
OUT_DIR = PROJECT / 'results/gnis_validation'
OUT_DIR.mkdir(exist_ok=True)
BOUNDS = (-114.25, 35.45, -111.25, 37.05)
D = 1229                      # main-canyon depth (m), from the Earth vs Mars analysis
MOUTH_DEPTH = 0.5 * D         # a tributary's mouth lies inside the deep main canyon
NEAR_M = 500                  # a detection within 500 m of the mouth or of the mouth-head line counts
MAIN_CANYON = {'Grand Canyon', 'Marble Canyon', 'Upper Granite Gorge', 'Middle Granite Gorge',
               'Lower Granite Gorge', 'Muav Gorge', 'Granite Gorge'}

# 1. Download the official Arizona names file (public USGS data, pipe-delimited text)
listing = requests.get('https://prd-tnm.s3.amazonaws.com/',
                       params={'list-type': '2', 'prefix': 'StagedProducts/GeographicNames/DomesticNames/'}, timeout=60).text
keys = (re.findall(r'<Key>([^<]*_AZ_[^<]*Text[^<]*\.zip)</Key>', listing)
        or re.findall(r'<Key>([^<]*_AZ[^<]*\.zip)</Key>', listing))
if not keys:
    raise RuntimeError('Arizona GNIS text file not found in the USGS listing.')
url = 'https://prd-tnm.s3.amazonaws.com/' + keys[0]
print('GNIS source:', url)
archive = zipfile.ZipFile(io.BytesIO(requests.get(url, timeout=300).content))
txt = [n for n in archive.namelist() if n.lower().endswith('.txt')][0]
names = pd.read_csv(archive.open(txt), sep='|', dtype=str, encoding='utf-8')
names.columns = [c.strip().lower() for c in names.columns]
for column in ['prim_lat_dec', 'prim_long_dec', 'source_lat_dec', 'source_long_dec']:
    names[column] = pd.to_numeric(names.get(column), errors='coerce')

# 2. Named valleys (GNIS class "Valley": canyons, washes, gulches) inside the Grand Canyon region
valleys = names[(names.feature_class == 'Valley') & names.prim_long_dec.between(BOUNDS[0], BOUNDS[2])
                & names.prim_lat_dec.between(BOUNDS[1], BOUNDS[3]) & ~names.feature_name.isin(MAIN_CANYON)].copy()

# 3. Keep only tributaries of the Grand Canyon: the mouth must sit inside the deep main canyon
mouths = gpd.GeoSeries(gpd.points_from_xy(valleys.prim_long_dec, valleys.prim_lat_dec), crs=4326).to_crs('EPSG:32612')
with rasterio.open(DEPTH) as src:
    # a mouth point can fall a pixel off the floor: take the deepest value within 400 m
    deep = maximum_filter(src.read(1), size=5)
    rows, cols = rowcol(src.transform, mouths.x.to_numpy(), mouths.y.to_numpy())
    rows = np.clip(np.asarray(rows), 0, deep.shape[0] - 1)
    cols = np.clip(np.asarray(cols), 0, deep.shape[1] - 1)
    values = deep[rows, cols]
valleys['mouth_depth_m'] = values.round()
reference = valleys[valleys.mouth_depth_m >= MOUTH_DEPTH].copy()
print(f'Named valleys in region: {len(valleys)} | named tributaries of the Grand Canyon (mouth >= {MOUTH_DEPTH:.0f} m deep): {len(reference)}')

# 4. Compare with the detections (all in UTM 12N metres)
detections = gpd.read_file(DETECTIONS).to_crs('EPSG:32612')
union = detections.union_all()
rows_out = []
for item in reference.itertuples():
    mouth = gpd.GeoSeries([Point(item.prim_long_dec, item.prim_lat_dec)], crs=4326).to_crs('EPSG:32612').iloc[0]
    has_head = pd.notna(item.source_long_dec) and item.source_long_dec != 0
    line = mouth
    if has_head:
        head = gpd.GeoSeries([Point(item.source_long_dec, item.source_lat_dec)], crs=4326).to_crs('EPSG:32612').iloc[0]
        line = LineString([mouth, head])
    distance = union.distance(line)
    nearest = detections.distance(line).idxmin()
    rows_out.append({'feature_id': item.feature_id, 'name': item.feature_name, 'mouth_lon': item.prim_long_dec,
                     'mouth_lat': item.prim_lat_dec, 'has_head_point': has_head, 'mouth_depth_m': item.mouth_depth_m,
                     'distance_m': round(float(distance)), 'found': bool(distance <= NEAR_M),
                     'nearest_detection': detections.loc[nearest, 'detection_id']})
check = pd.DataFrame(rows_out).sort_values('name')
check.to_csv(OUT_DIR / 'gnis_named_tributaries_check.csv', index=False)

# 5. Give every detection the name of the official canyon it contains (nearest within 500 m)
labels = []
for detection in detections.itertuples():
    candidates = []
    for item in check[check.distance_m <= NEAR_M].itertuples():
        if item.nearest_detection == detection.detection_id:
            candidates.append((item.distance_m, item.name))
    labels.append(sorted(candidates)[0][1] if candidates else '')
detections['gnis_name'] = labels
named = detections.to_crs(4326)
named.to_file(OUT_DIR / 'grand_canyon_detections_named.geojson', driver='GeoJSON')
named.to_file(PROJECT / 'results/grand_canyon/grand_canyon_detections_named.geojson', driver='GeoJSON')

found = check.found.mean() * 100
summary = {
    'gnis_source': url, 'named_tributaries': int(len(check)), 'found': int(check.found.sum()),
    'recall_percent': round(found, 1), 'detections': int(len(detections)),
    'detections_with_official_name': int((detections.gnis_name != '').sum()), 'distance_rule_m': NEAR_M,
    'mouth_depth_rule_m': MOUTH_DEPTH,
}
(OUT_DIR / 'gnis_summary.json').write_text(json.dumps(summary, indent=2))

print('\n===== Independent check with USGS GNIS =====')
print(f"Official named tributary canyons: {summary['named_tributaries']}")
print(f"Found by CanyonScope (within {NEAR_M} m): {summary['found']} ({summary['recall_percent']}%)")
print(f"Detections that received an official name: {summary['detections_with_official_name']} of {summary['detections']}")
print('\nFound (first 25):', ', '.join(check[check.found].name.head(25)))
print('\nMissed (all):', ', '.join(check[~check.found].name))
print('\nSaved to:', OUT_DIR)


## 2. Error analysis: recall by reach of the river and by type of name

In [ ]:
# Step V2: why were some official canyons missed? (error analysis, reads the V1 table only)
import re
from pathlib import Path

import pandas as pd

OUT_DIR = REPO_DIR / 'results/gnis_validation'
check = pd.read_csv(OUT_DIR / 'gnis_named_tributaries_check.csv')


def reach(row):
    if row.mouth_lon > -111.95 and row.mouth_lat > 36.20:
        return '1 Marble Canyon (east)'
    if row.mouth_lon < -113.50:
        return '3 Western canyon / Lake Mead'
    return '2 Central Grand Canyon'


def kind(name):
    if re.search(r'\b(Canyon|Gorge|Chasm)\b', name):
        return 'Canyon / Gorge'
    return 'Wash, Valley, Draw, Hollow, other'


check['reach'] = check.apply(reach, axis=1)
check['kind'] = check.name.map(kind)


def table(column):
    out = check.groupby(column).agg(named=('found', 'size'), found=('found', 'sum'))
    out['recall_%'] = (out.found / out.named * 100).round(1)
    return out


print('===== Recall by reach of the river =====')
print(table('reach').to_string())
print('\n===== Recall by type of name =====')
print(table('kind').to_string())
print('\n===== Recall for true canyons only, by reach =====')
canyons = check[check.kind == 'Canyon / Gorge']
out = canyons.groupby('reach').agg(named=('found', 'size'), found=('found', 'sum'))
out['recall_%'] = (out.found / out.named * 100).round(1)
print(out.to_string())
print(f'All true canyons: {canyons.found.sum()} of {len(canyons)} ({canyons.found.mean() * 100:.1f}%)')

print('\n===== Sensitivity to the distance rule =====')
for metres in [250, 500, 1000, 2000]:
    print(f'within {metres:>4} m: {(check.distance_m <= metres).mean() * 100:5.1f}% of {len(check)} found')

check.to_csv(OUT_DIR / 'gnis_named_tributaries_check.csv', index=False)
print('\nSaved (with reach and type columns):', OUT_DIR / 'gnis_named_tributaries_check.csv')
